# CogMem — BigCodeBench-Hard Evaluation

Run models on BigCodeBench-Hard (148 tasks). Collect episodes with success/failure + model reasoning.

**Models:** Set `MODEL` variable in Cell 6 to switch between:
- `qwen2.5:3b` — base model (baseline)
- `cogmem-qwen-bigcode` — CogMem DoRA model (after Phase 2 training)

**Flow:** Cells 1-6 sequentially. If notebook restarts, run 1-4, then 6 (auto-resumes).

**Resume-safe:** Checkpoint saves after every task.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [5]:
# Cell 2: Install system deps + Ollama (DO NOT touch torch)
!apt-get update -qq && apt-get install -y -qq zstd cmake build-essential > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
!pip install "transformers==4.43.4" "peft==0.12.0" "accelerate==0.33.0" "datasets>=2.20" "huggingface-hub>=0.24" pyyaml -q
!pip install openai -q

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%     3.2%
>>> Adding ollama user to render group...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cogmem 0.1.0 requires openai>=1.0, which is not installed.
cogmem 0.1.0 requires together>=1.0, which is not inst

In [6]:
# Cell 3: Start Ollama + pull model
import subprocess, time, os
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!ollama pull qwen2.5:3b
print("Ollama + Qwen2.5:3b ready!")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 5ee4f07cdb9b: 100% ▕██████████████████▏ 1.9 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling b5c0e5cf74cf: 100% ▕██████████████████▏ 7.4 KB                         
pulling 161ddde4c9cd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
Ollama + Qwen2.5:3b ready!


In [ ]:
# Cell 4: Clone CogMem + load BigCodeBench-Hard dataset
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import cogmem
print(f"cogmem loaded from {cogmem.__file__}")

from datasets import load_dataset

# BigCodeBench-Hard: 148 tasks (the standard eval subset)
ds = load_dataset("bigcode/bigcodebench-hard", split="v0.1.4")
print(f"BigCodeBench-Hard: {len(ds)} tasks")

# Convert to list of dicts
tasks = []
for item in ds:
    tasks.append({
        "task_id": item["task_id"],
        "instruct_prompt": item.get("instruct_prompt", ""),
        "complete_prompt": item.get("complete_prompt", ""),
        "test": item.get("test", ""),
        "canonical_solution": item.get("canonical_solution", ""),
        "entry_point": item.get("entry_point", ""),
    })

import json
with open("/notebooks/bigcodebench_hard_tasks.jsonl", "w") as f:
    for t in tasks:
        f.write(json.dumps(t) + "\n")
print(f"Saved {len(tasks)} tasks to /notebooks/bigcodebench_hard_tasks.jsonl")

In [ ]:
# Cell 5: Quick sanity check — run 3 tasks to verify pipeline
import importlib
import cogmem.benchmarks.bigcodebench.evaluator
import cogmem.benchmarks.bigcodebench.prompts
importlib.reload(cogmem.benchmarks.bigcodebench.evaluator)
importlib.reload(cogmem.benchmarks.bigcodebench.prompts)

from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

for task in tasks[:3]:
    messages = format_messages(task, use_instruct=True)
    resp = client.chat.completions.create(
        model="qwen2.5:3b", messages=messages,
        max_tokens=2048, temperature=0,
    )
    response = resp.choices[0].message.content
    code = extract_code(response)
    result = evaluate_solution(task, code, timeout=30, mode="subprocess")
    status = "PASS" if result["passed"] else "FAIL"
    print(f"{task['task_id']}: {status}")
    if not result["passed"] and result.get("error"):
        print(f"  Error: {result['error'][:200]}")
    print()

print("Sanity check done!")

In [ ]:
# Cell 6: Run BigCodeBench-Hard evaluation (resume-safe)
# ~15-20 min for 148 tasks on A4000
#
# CHANGE THIS to switch models:
MODEL = "qwen2.5:3b"  # baseline
# MODEL = "cogmem-qwen-bigcode"  # after Phase 2 training

import json, time
from pathlib import Path
from openai import OpenAI
from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

# Checkpoint per model — so you can run both without overwriting
CHECKPOINT = f"/notebooks/bigcode_hard_{MODEL.replace(':', '_').replace('-', '_')}.jsonl"

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Resume: load completed task IDs
completed_ids = set()
episodes = []
if Path(CHECKPOINT).exists():
    with open(CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                ep = json.loads(line)
                completed_ids.add(ep["task_id"])
                episodes.append(ep)
            except json.JSONDecodeError:
                print("Warning: skipping truncated line in checkpoint")
                break
    print(f"Resuming: {len(completed_ids)} tasks already done")

# Load tasks
tasks = []
with open("/notebooks/bigcodebench_hard_tasks.jsonl") as f:
    for line in f:
        tasks.append(json.loads(line.strip()))

remaining = [t for t in tasks if t["task_id"] not in completed_ids]
total = len(tasks)
done = len(completed_ids)
passed = sum(1 for ep in episodes if ep["success"])
start_time = time.time()

print(f"Model: {MODEL}")
print(f"Total: {total}, Done: {done}, Remaining: {len(remaining)}")
print(f"Current pass rate: {passed}/{done} = {passed/max(done,1):.1%}")
print("=" * 60)

for i, task in enumerate(remaining):
    try:
        messages = format_messages(task, use_instruct=True)
        resp = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=2048, temperature=0,
        )
        response = resp.choices[0].message.content
        code = extract_code(response)
        result = evaluate_solution(task, code, timeout=30, mode="subprocess")

        episode = {
            "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
            "task_id": task["task_id"],
            "task_type": "bigcodebench_hard",
            "task_description": task.get("instruct_prompt", task.get("complete_prompt", "")),
            "script": response,
            "generated_code": code,
            "success": result["passed"],
            "q_value": 1.0 if result["passed"] else -1.0,
            "error": result.get("error"),
            "entry_point": task.get("entry_point", ""),
            "model": MODEL,
            "timestamp": time.time(),
        }
    except Exception as e:
        episode = {
            "episode_id": f"bigcode_{task['task_id'].replace('/', '_')}_{int(time.time())}",
            "task_id": task["task_id"],
            "task_type": "bigcodebench_hard",
            "task_description": task.get("instruct_prompt", ""),
            "script": "",
            "generated_code": "",
            "success": False,
            "q_value": -1.0,
            "error": str(e),
            "entry_point": task.get("entry_point", ""),
            "model": MODEL,
            "timestamp": time.time(),
        }

    episodes.append(episode)
    done += 1
    if episode["success"]:
        passed += 1

    with open(CHECKPOINT, "a", encoding="utf-8") as f:
        f.write(json.dumps(episode, ensure_ascii=False) + "\n")

    status = "PASS" if episode["success"] else "FAIL"
    elapsed = time.time() - start_time
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    eta = (len(remaining) - i - 1) / rate * 3600 if rate > 0 else 0

    if (i + 1) % 10 == 0 or i < 5:
        print(f"[{done}/{total}] {task['task_id']}: {status} | "
              f"Pass: {passed}/{done} ({passed/done:.1%}) | "
              f"Rate: {rate:.0f}/hr | ETA: {eta/60:.0f}m")

print("=" * 60)
print(f"DONE! {MODEL}: {passed}/{done} passed ({passed/done:.1%})")
print(f"Checkpoint: {CHECKPOINT}")

In [ ]:
# Cell 7: Show results for all models evaluated
import json
from pathlib import Path
from collections import Counter

print("=" * 60)
print("BigCodeBench-Hard Results")
print("=" * 60)

# Find all checkpoint files
for ckpt in sorted(Path("/notebooks").glob("bigcode_hard_*.jsonl")):
    episodes = []
    with open(ckpt, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                episodes.append(json.loads(line))
    
    if not episodes:
        continue
    
    total = len(episodes)
    passed = sum(1 for ep in episodes if ep["success"])
    model = episodes[0].get("model", ckpt.stem)
    
    print(f"\n  {model}: {passed}/{total} ({passed/total:.1%})")
    
    # Error breakdown
    errors = Counter()
    for ep in episodes:
        if not ep["success"] and ep.get("error"):
            err = ep["error"][:100]
            if "Timeout" in err:
                errors["Timeout"] += 1
            elif "SyntaxError" in err:
                errors["SyntaxError"] += 1
            elif "ImportError" in err or "ModuleNotFoundError" in err:
                errors["ImportError"] += 1
            else:
                errors["Other"] += 1
    
    for err_type, count in errors.most_common(3):
        print(f"    {err_type}: {count}")

print("\n" + "=" * 60)

In [ ]:
# Cell 8: (Optional) Build training data from Hard episodes
# Only needed if you want to retrain on Hard-specific episodes
import json
from pathlib import Path

# Combine all model checkpoints into one memory bank
all_episodes = []
for ckpt in sorted(Path("/notebooks").glob("bigcode_hard_*.jsonl")):
    with open(ckpt, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                all_episodes.append(json.loads(line))

if all_episodes:
    MB_OUTPUT = "/notebooks/CogMem/results/memory_bank_bigcode_hard.json"
    Path(MB_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    with open(MB_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(all_episodes, f, indent=2, ensure_ascii=False)
    
    passed = sum(1 for ep in all_episodes if ep["success"])
    print(f"Saved {len(all_episodes)} episodes to {MB_OUTPUT}")
    print(f"Overall: {passed}/{len(all_episodes)} ({passed/len(all_episodes):.1%})")
else:
    print("No Hard episodes found. Run Cell 6 first.")